# 01 - Reading SPINE Output

Goal: open a reconstructed SPINE HDF5 file, inspect the high-level object hierarchy, and connect particle/interactions fields to analysis questions.

This notebook is the highest-priority hands-on material for the short session. It is adapted from the 2026 workshop HDF5 readback material, with inference moved out of scope.

## Runtime contract

Run inside `ghcr.io/deeplearnphysics/spine:latest`.

Set the file path in the first code cell:

```python
DATA_PATH = "/path/to/reconstructed_spine_file.h5"
DETECTOR = "icarus"  # or None
```

The file should contain reconstructed particles/interactions and, for validation cells, truth objects.
Use `DETECTOR = None` when detector geometry is not needed for the exercise.

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from spine.driver import Driver

# Edit these two lines for the sample you want to inspect.
# DATA_PATH may be a single HDF5 file or a glob such as "/path/to/reco/*.h5".
DATA_PATH = "CHANGE_ME/reconstructed_spine_file.h5"

# Use None if no detector geometry is needed. Common examples:
# "icarus", "sbnd", "2x2", "nd-lar", "protodune-sp", "protodune-vd"
DETECTOR = None

CONFIG_CANDIDATES = [
    Path("../config/read_spine_hdf5.yaml"),
    Path("tutorials/config/read_spine_hdf5.yaml"),
]
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

if DATA_PATH.startswith("CHANGE_ME"):
    raise RuntimeError(
        "Set DATA_PATH at the top of this notebook to the tutorial HDF5 file or file glob."
    )
if CONFIG_PATH is None:
    raise RuntimeError("Could not find tutorials/config/read_spine_hdf5.yaml")

cfg_text = CONFIG_PATH.read_text().replace("DATA_PATH", DATA_PATH)
cfg = yaml.safe_load(cfg_text)
if DETECTOR:
    cfg["geo"] = {"detector": DETECTOR}
driver = Driver(cfg)
print(f"Opened {DATA_PATH}")
print(f"Entries: {len(driver)}")
if DETECTOR:
    print(f"Detector geometry: {DETECTOR}")


## Where to look things up

This tutorial deliberately pauses on small snippets. When a field or class is unfamiliar, use three complementary tools:

- Python introspection: `help(obj)`, `dir(obj)`, `obj.as_dict().keys()`
- The SPINE API browser: https://spine.readthedocs.io
- Source/config examples in `spine-prod`

The point is not to memorize every field. The point is to learn how to inspect the object model without guessing.

In [ ]:
ENTRY = 0
data = driver.process(entry=ENTRY)
print(data.keys())

reco_particles = data.get("reco_particles", [])
truth_particles = data.get("truth_particles", [])
reco_interactions = data.get("reco_interactions", [])
truth_interactions = data.get("truth_interactions", [])

print(f"Reco particles: {len(reco_particles)}")
print(f"Truth particles: {len(truth_particles)}")
print(f"Reco interactions: {len(reco_interactions)}")
print(f"Truth interactions: {len(truth_interactions)}")

## Inspect one object

SPINE objects expose Python attributes and usually an `as_dict()` method. Start by looking at one reconstructed particle and one reconstructed interaction.

Pause on the line `d = obj.as_dict()`: it is the fastest way to discover the analysis-facing fields saved in this file.

In [ ]:
def compact_dict(obj, max_items=40):
    if hasattr(obj, "as_dict"):
        d = obj.as_dict()
    else:
        d = vars(obj)
    rows = []
    for key, value in list(d.items())[:max_items]:
        if isinstance(value, np.ndarray):
            rows.append((key, f"array shape={value.shape} dtype={value.dtype}"))
        else:
            rows.append((key, repr(value)[:120]))
    return pd.DataFrame(rows, columns=["field", "value"])

if reco_particles:
    display(compact_dict(reco_particles[0]))
if reco_interactions:
    display(compact_dict(reco_interactions[0]))

In [ ]:
# Try this on one object during the tutorial.
# help(reco_particles[0])
# dir(reco_particles[0])
# reco_particles[0].as_dict().keys()

In [ ]:
PID_LABELS = {
    -1: "unknown",
    0: "photon",
    1: "electron",
    2: "muon",
    3: "pion",
    4: "proton",
    5: "kaon",
}

SHAPE_LABELS = {
    -1: "unknown",
    0: "shower",
    1: "track",
    2: "michel",
    3: "delta",
    4: "low-energy",
}

def particle_row(p):
    return {
        "id": getattr(p, "id", None),
        "interaction_id": getattr(p, "interaction_id", None),
        "pid": getattr(p, "pid", None),
        "pid_label": PID_LABELS.get(getattr(p, "pid", None), getattr(p, "pid", None)),
        "shape": getattr(p, "shape", None),
        "shape_label": SHAPE_LABELS.get(getattr(p, "shape", None), getattr(p, "shape", None)),
        "primary": getattr(p, "is_primary", None),
        "size": getattr(p, "size", None),
        "depositions_sum": getattr(p, "depositions_sum", np.nan),
        "start_point": getattr(p, "start_point", None),
        "end_point": getattr(p, "end_point", None),
    }

particle_df = pd.DataFrame([particle_row(p) for p in reco_particles])
particle_df

In [ ]:
def interaction_row(ia):
    particles = getattr(ia, "particles", [])
    primary_particles = [p for p in particles if getattr(p, "is_primary", False)]
    return {
        "id": getattr(ia, "id", None),
        "nu_id": getattr(ia, "nu_id", None),
        "size": getattr(ia, "size", None),
        "vertex": getattr(ia, "vertex", None),
        "n_particles": len(particles),
        "n_primary": len(primary_particles),
        "primary_pids": [getattr(p, "pid", None) for p in primary_particles],
        "topology": getattr(ia, "topology", None),
    }

interaction_df = pd.DataFrame([interaction_row(ia) for ia in reco_interactions])
interaction_df

## Live exercise

Pick one interaction and answer:

- Which primary particles does SPINE reconstruct?
- Is the vertex close to the visually obvious interaction point?
- Which fields would you trust for a first-pass analysis selection, and which require validation?

In [ ]:
from spine.vis.out import Drawer

drawer = Drawer(data)
fig = drawer.get("interactions", "id")
fig.show()

## Inference belongs in `spine-prod`

This short tutorial starts from HDF5 output. For real production, `spine-prod` should own the validated full-chain inference configs, model weights, campaign metadata, and file bookkeeping. That lets analysis notebooks stay focused on object interpretation and validation.

## Offline extensions

- Repeat the field inspection for a second detector sample and list which object fields are detector-dependent.
- Build a small field glossary for the 10 particle/interactions attributes your analysis will use.
- Compare the same event in this notebook and Spinal Tap, then record which table fields explain the visual topology.